# AYA Edge 0.2 — First Real Training

Kaggle/T4 entry point. This notebook refuses to train until a separately reviewed corpus and the matching 32,768-token artifact are attached as Kaggle input. A successful loss reduction is not evidence of agent competence.

In [ ]:
# Configure these paths to an attached Kaggle Dataset; no corpus is downloaded here.
REPO_URL = 'https://github.com/hygienhub03-dev/Aya-Edge-0.1.git'
REVISION = 'main'  # replace with the reviewed commit SHA for a recorded run
INPUT_DIR = '/kaggle/input/aya-edge-approved-language-v1'
OUTPUT_DIR = '/kaggle/working/aya-edge-output'
MAX_STEPS = 100  # short real-learning budget; raise only after reviewing cost
RESUME_CHECKPOINT = None  # e.g. '/kaggle/input/aya-edge-checkpoint/last.pt'
assert MAX_STEPS > 0
SMOKE_TEST = MAX_STEPS <= 2
print({'smoke_test': SMOKE_TEST, 'max_steps': MAX_STEPS})

In [ ]:
import json, os, pathlib, platform, subprocess, sys, hashlib
import torch
print({'python': sys.version, 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
if not torch.cuda.is_available():
    print('Warning: no GPU detected; this will be much slower than a T4 run.')
WORK = pathlib.Path('/kaggle/working/aya-edge')
if not WORK.exists(): subprocess.run(['git', 'clone', '--depth', '1', '--branch', REVISION, REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('repository revision:', HEAD)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data]'], check=True)

In [ ]:
# Preflight: require artifacts generated from the approved, bounded corpus.
from pathlib import Path
input_dir, out_dir = Path(INPUT_DIR), Path(OUTPUT_DIR)
tokenizer = input_dir / 'tokenizer.json'
tokenizer_manifest = input_dir / 'tokenizer.manifest.json'
corpus_manifest = input_dir / 'manifest.json'
approval = input_dir / 'tokenizer_approval.json'
train_jsonl, validation_jsonl = input_dir / 'train.jsonl', input_dir / 'validation.jsonl'
required = [tokenizer, tokenizer_manifest, corpus_manifest, approval, train_jsonl, validation_jsonl]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Attach the approved corpus/artifact dataset; missing: ' + ', '.join(missing))
approval_data = json.loads(approval.read_text())
if approval_data.get('approved') is not True: raise PermissionError('Corpus is not explicitly approved for this run.')
if not approval_data.get('license_audit_sha256') or not approval_data.get('source_registry_revision'): raise ValueError('Approval record is not pinned.')
manifest_data = json.loads(corpus_manifest.read_text())
if manifest_data.get('metadata', {}).get('license_audit_sha256') != approval_data['license_audit_sha256']: raise ValueError('Corpus manifest does not match approval record.')
print({'approval': approval_data['scope'], 'records': manifest_data.get('records')})
subprocess.run([sys.executable, 'scripts/validate_bpe_tokenizer.py', '--tokenizer', str(tokenizer), '--manifest', str(tokenizer_manifest), '--config', 'configs/aya_edge_0_2_language.yaml'], check=True)

In [ ]:
# The training entry point constructs the authoritative 0.2 MLA-inspired MoE model, reports total/active parameters, packs sequences, uses AMP and gradient accumulation, measures validation loss and saves resumable checkpoints.
out_dir.mkdir(parents=True, exist_ok=True)
checkpoint = out_dir / 'aya_edge_0_2_language.pt'
metrics = out_dir / 'metrics.jsonl'
command = [sys.executable, 'scripts/train_language_model.py', '--config', 'configs/aya_edge_0_2_language.yaml', '--tokenizer', str(tokenizer), '--input-jsonl', str(train_jsonl), '--validation-jsonl', str(validation_jsonl), '--output', str(checkpoint), '--metrics', str(metrics), '--max-steps', str(MAX_STEPS), '--allow-dirty']
if RESUME_CHECKPOINT: command.extend(['--resume', RESUME_CHECKPOINT])
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
# Post-training sample: illustrative only, not an agent-capability evaluation.
import yaml
from aya_edge.config import AyaEdgeConfig
from aya_edge.model import AyaEdgeForCausalLM
from aya_edge.tokenizer.bpe import ProductionBPETokenizer
cfg = AyaEdgeConfig(**yaml.safe_load(Path('configs/aya_edge_0_2_language.yaml').read_text())['model'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_model = AyaEdgeForCausalLM(cfg).to(device)
sample_model.load_state_dict(torch.load(checkpoint, map_location=device)['model'])
sample_tokenizer = ProductionBPETokenizer.load(tokenizer)
prompt = 'The purpose of reliable software is'
ids = torch.tensor([sample_tokenizer.encode(prompt)], device=device)
generated = sample_model.generate(ids, max_new_tokens=64, eos_token_id=3, temperature=0.8, top_k=40)[0].tolist()
print(sample_tokenizer.decode(generated))

In [ ]:
# Export reproducibility metadata and checksums to persistent Kaggle output.
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()
summary = {'git_revision': HEAD, 'seed': 42, 'max_steps': MAX_STEPS, 'smoke_test': SMOKE_TEST, 'tokenizer_sha256': sha256(tokenizer), 'corpus_manifest_sha256': sha256(corpus_manifest), 'checkpoint': str(checkpoint), 'checkpoint_sha256': sha256(checkpoint), 'metrics': str(metrics), 'gpu_max_memory_bytes': int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0}
(out_dir / 'run_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
print('For a generation sample, load the saved checkpoint with the same validated tokenizer; do not treat it as an agent benchmark.')